In [ ]:
!pip install easyocr ultralytics faiss-cpu rapidfuzz

In [ ]:
import pandas as pd
import numpy as np
import faiss
import json
import os
import re
import cv2
import matplotlib.pyplot as plt
import easyocr
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer
from ultralytics import YOLO
import glob
import time

# ==========================
# 📍 КОНФИГУРАЦИЯ
# ==========================
# Пути к файлам
KB_CSV_PATH = "/content/drive/MyDrive/Lenta Tech Life Hack/merged_dataset.csv"         # База данных
VIDEO_PATH = "/content/drive/MyDrive/Lenta Tech Life Hack/Данные/25_2-10/25_2-10.mp4"  # Ваш видеофайл
OUTPUT_CSV = "video_recognition_results.csv"

YOLO_MODEL_1_PATH = "/content/drive/MyDrive/Lenta Tech Life Hack/model_1/best.pt"      # YOLO_1
YOLO_MODEL_2_PATH = "/content/drive/MyDrive/Lenta Tech Life Hack/model_2/best.pt"      # YOLO_2

FAISS_PATH = "/content/drive/MyDrive/Lenta Tech Life Hack/model_2/price_tag_hybrid.faiss"        # embeddings
META_PATH = "/content/drive/MyDrive/Lenta Tech Life Hack/model_2/price_tag_hybrid_meta.json"

# Параметры
MATCH_THRESHOLD = 0.55
YOLO_CONF_THRESH_1 = 0.3
YOLO_CONF_THRESH_2 = 0.25

# Настройки видео
FRAME_SKIP = 10          # Пропуск кадров (10 = обрабатываем каждый 10-й кадр)
BLUR_THRESHOLD = 80.0    # Порог резкости (ниже — кадр слишком размыт)
DUP_WINDOW = 30          # Окно дедупликации (кадры). Не записывать тот же товар чаще чем раз в N кадров.

# ==========================
# 🧠 1. ПОДГОТОВКА ДАННЫХ И ИНДЕКСОВ
# ==========================
def load_kb_and_indexes():
    price_index = {}

    if os.path.exists(FAISS_PATH) and os.path.exists(META_PATH):
        print("💾 Загрузка готовых индексов из кэша...")
        index = faiss.read_index(FAISS_PATH)
        with open(META_PATH, "r", encoding="utf-8") as f:
            metadata = json.load(f)
        df = pd.DataFrame(metadata)
    else:
        print("📖 Чтение CSV и построение индексов...")
        df = pd.read_csv(KB_CSV_PATH)
        df = df.fillna("").astype(str)

        print("🤖 Загрузка BGE-M3...")
        embed_model = SentenceTransformer("BAAI/bge-m3", device="cpu")

        composite_texts = []
        for _, row in df.iterrows():
            name = row.get("product_name", "")
            discount = str(row.get("discount_amount", ""))
            p_card = str(row.get("price_card", ""))
            text = f"{name} | Скидка: {discount} | Цена по карте: {p_card}"
            composite_texts.append(text)

        print("⚡ Векторизация...")
        embeddings = embed_model.encode(composite_texts, normalize_embeddings=True, batch_size=32)

        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings.astype(np.float32))

        faiss.write_index(index, FAISS_PATH)
        with open(META_PATH, "w", encoding="utf-8") as f:
            json.dump(df.to_dict(orient="records"), f, ensure_ascii=False)

    print("🏗 Построение индекса по ценам...")
    for idx, row in df.iterrows():
        prices_to_index = []
        try:
            p_card = float(str(row.get("price_card", "0")).replace(',', '.'))
            if p_card > 0: prices_to_index.append(int(p_card))
        except: pass
        try:
            p_def = float(str(row.get("price_default", "0")).replace(',', '.'))
            if p_def > 0 and int(p_def) not in prices_to_index:
                prices_to_index.append(int(p_def))
        except: pass

        for p_int in prices_to_index:
            if p_int not in price_index:
                price_index[p_int] = []
            price_index[p_int].append(idx)

    print(f"✅ Индексы готовы. Товаров: {len(df)}. Уникальных цен: {len(price_index)}")
    return df, index, price_index

# ==========================
# 🔄 2. ДЕТЕКЦИЯ ЦЕННИКОВ (МОДЕЛЬ 1)
# ==========================
def detect_all_and_align_price_tags(img, yolo_model_1):
    """Находит все ценники и возвращает данные (оригинальные координаты + повернутый кроп)"""
    results = yolo_model_1.predict(img, conf=YOLO_CONF_THRESH_1, verbose=False)
    boxes = results[0].boxes

    tags_list = []
    if len(boxes) == 0:
        return tags_list

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        roi = img[y1:y2, x1:x2].copy()
        if roi.size == 0: continue

        # Поворот налево на 90 градусов
        roi_aligned = cv2.rotate(roi, cv2.ROTATE_90_COUNTERCLOCKWISE)

        tags_list.append({
            "tag_id": i,
            "original_bbox": (x1, y1, x2, y2), # <-- ВАЖНО: сохраняем координаты на исходном кадре
            "roi_aligned": roi_aligned
        })
    return tags_list

# ==========================
#  3. OCR И ПРЕПРОЦЕССИНГ
# ==========================
def preprocess_for_ocr(image, method='standard'):
    if image is None: return None
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()

    scale = 3.0
    h, w = gray.shape
    resized = cv2.resize(gray, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC)

    if method == 'price_red':
        binary1 = cv2.adaptiveThreshold(resized, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
        binary1 = cv2.bitwise_not(binary1)
        blur = cv2.GaussianBlur(resized, (5,5), 0)
        _, binary2 = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        enhanced = cv2.convertScaleAbs(resized, alpha=2.0, beta=0)
        return [binary1, binary2, enhanced]
    else:
        enhanced = cv2.convertScaleAbs(resized, alpha=1.5, beta=20)
        return [enhanced]

def clean_ocr_text(text):
    if not text: return ""
    text = ' '.join(text.split())
    text = re.sub(r'[^\w\s\-\.\(\)\/%]', ' ', text)
    return text.strip()

def parse_price_strict(text):
    if not text: return []
    candidates = set()
    text_clean = re.sub(r'[^\d\.,\s]', ' ', text)

    matches_std = re.findall(r'(\d{1,4})[\s,\.](\d{2})\b', text_clean)
    for m in matches_std:
        val = float(f"{m[0]}.{m[1]}")
        if 10 <= val <= 100000: candidates.add(val)

    matches_long = re.findall(r'\b(\d{5,7})\b', text_clean)
    for m in matches_long:
        num = int(m)
        cents, rubles = num % 100, num // 100
        if 10 <= rubles <= 100000 and cents < 100:
            candidates.add(float(f"{rubles}.{cents:02d}"))

    matches_int = re.findall(r'\b(\d{2,4})\b', text_clean)
    for m in matches_int:
        val = int(m)
        if 50 <= val <= 100000: candidates.add(float(val))

    return sorted(list(candidates), reverse=True)

def parse_discount(text):
    if not text: return ""
    match = re.search(r'[-]?\s*(\d{1,2})\s*[%oO]?', text)
    if match:
        val = int(match.group(1))
        if 5 <= val <= 90: return f"-{val}%"
    return ""

# ==========================
# 🔍 4. ДЕТЕКЦИЯ ПОЛЕЙ (МОДЕЛЬ 2)
# ==========================
def extract_fields_with_yolo(img, yolo_model_2, ocr_reader):
    results = yolo_model_2.predict(img, conf=YOLO_CONF_THRESH_2, verbose=False)
    boxes = results[0].boxes
    class_map = results[0].names

    detections = {'наименование': [], 'скидка': [], 'цена': []}
    for box in boxes:
        cls_id = int(box.cls[0])
        cls_name = class_map.get(cls_id, "")
        if cls_name in detections:
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            detections[cls_name].append((x1, y1, x2, y2, float(box.conf[0])))

    roi_texts = {'наименование': '', 'скидка': '', 'цена': ''}
    h, w, _ = img.shape

    for cls, box_list in detections.items():
        if not box_list: continue
        box_list.sort(key=lambda x: x[4], reverse=True)
        x1, y1, x2, y2, _ = box_list[0]

        pad = 15
        roi = img[max(0, y1-pad):min(h, y2+pad), max(0, x1-pad):min(w, x2+pad)]
        if roi.size == 0: continue

        methods = preprocess_for_ocr(roi, method='price_red' if cls == 'цена' else 'standard')
        all_texts = []
        for method_img in methods:
            if method_img is not None:
                ocr_results = ocr_reader.readtext(method_img, detail=1)
                texts = [t for _, t, c in ocr_results if c > 0.1]
                all_texts.extend(texts)

        if all_texts:
            combined = " ".join(all_texts)
            roi_texts[cls] = clean_ocr_text(combined)

    return roi_texts, detections

# ==========================
# 🎯 5. ГИБРИДНЫЙ ПОИСК
# ==========================
def hybrid_search(df, index, price_index, ocr_data, embed_model):
    ocr_name = ocr_data['наименование']
    ocr_price_candidates = parse_price_strict(ocr_data['цена'])
    ocr_disc = parse_discount(ocr_data['скидка'])

    best_match = None
    best_score = 0.0

    # ШАГ 1: Поиск по цене
    if ocr_price_candidates:
        target_price = int(ocr_price_candidates[0])
        candidate_indices = set()
        for delta in range(-3, 4):
            lookup_price = target_price + delta
            if lookup_price in price_index:
                for idx in price_index[lookup_price]:
                    candidate_indices.add(idx)

        if candidate_indices:
            for idx in list(candidate_indices)[:300]:
                row = df.iloc[idx]
                db_name = str(row.get("product_name", ""))

                score_ratio = fuzz.partial_ratio(ocr_name.lower(), db_name.lower()) / 100.0
                score_token = fuzz.token_sort_ratio(ocr_name.lower(), db_name.lower()) / 100.0
                name_sim = max(score_ratio, score_token)

                disc_bonus = 0.0
                db_disc = str(row.get("discount_amount", ""))
                if ocr_disc and db_disc:
                    if re.sub(r'[^0-9]', '', ocr_disc) == re.sub(r'[^0-9]', '', db_disc):
                        disc_bonus = 0.15

                price_bonus = 0.0
                try:
                    db_price_card = float(str(row.get("price_card", "0")).replace(',', '.'))
                    if abs(db_price_card - ocr_price_candidates[0]) < 1.0:
                        price_bonus = 0.1
                except: pass

                final_score = 0.4 + (name_sim * 0.4) + disc_bonus + price_bonus
                if final_score > best_score:
                    best_score = final_score
                    best_match = {"row": row.to_dict(), "score": final_score, "method": "price_match"}

    # ШАГ 2: Семантический поиск
    if best_match is None and ocr_name:
        query_text = f"{ocr_name} | Скидка: {ocr_data['скидка']} | Цена по карте: {ocr_data['цена']}"
        q_vec = embed_model.encode([query_text], normalize_embeddings=True)[0]
        scores, ids = index.search(np.array([q_vec], dtype=np.float32), 30)

        for sim, idx in zip(scores[0], ids[0]):
            if idx != -1:
                row = df.iloc[idx]
                try:
                    db_price_card = float(str(row.get("price_card", "0")).replace(',', '.'))
                    if ocr_price_candidates and abs(db_price_card - ocr_price_candidates[0]) > 10:
                        continue
                except: pass

                disc_bonus = 0.0
                db_disc = str(row.get("discount_amount", ""))
                if ocr_disc and db_disc:
                    if re.sub(r'[^0-9]', '', ocr_disc) == re.sub(r'[^0-9]', '', db_disc):
                        disc_bonus = 0.1

                final_score = float(sim) * 0.8 + disc_bonus
                if final_score > best_score:
                    best_score = final_score
                    best_match = {"row": row.to_dict(), "score": final_score, "method": "semantic"}

    return best_match, best_score

# ==========================
# 🚀 ОСНОВНОЙ ПАЙПЛАЙН ВИДЕО
# ==========================
def process_video():
    print("🔄 Инициализация...")
    df, index, price_index = load_kb_and_indexes()
    embed_model = SentenceTransformer("BAAI/bge-m3", device="cpu")

    print("🤖 Загрузка моделей...")
    yolo_model_1 = YOLO(YOLO_MODEL_1_PATH)
    yolo_model_2 = YOLO(YOLO_MODEL_2_PATH)
    ocr_reader = easyocr.Reader(["ru", "en"], gpu=False, verbose=False)

    if not os.path.exists(VIDEO_PATH):
        print(f"❌ Видеофайл не найден: {VIDEO_PATH}")
        return

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print(" Не удалось открыть видео")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"🎥 Видео: {total_frames} кадров, {fps} FPS")

    # Словарь для дедупликации: {id_sku: last_seen_frame_index}
    recognized_products = {}
    results = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 🔥 НАЧИНАЕМ С ПЕРВОГО КАДРА, но с пропуском для скорости
        if frame_count % FRAME_SKIP != 0:
            frame_count += 1
            continue

        # 1. Проверка на размытие
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
        if blur_score < BLUR_THRESHOLD:
            frame_count += 1
            continue

        print(f"\n📷 Обработка кадра {frame_count} (Time: {int((frame_count/fps)*1000)} ms)...")

        # 2. Детекция ценников
        tags_data = detect_all_and_align_price_tags(frame, yolo_model_1)

        for tag_item in tags_data:
            original_bbox = tag_item['original_bbox'] # (x1, y1, x2, y2)
            aligned_tag = tag_item['roi_aligned']

            # 3. Детекция полей (Model 2)
            ocr_data, _ = extract_fields_with_yolo(aligned_tag, yolo_model_2, ocr_reader)

            if not ocr_data['наименование'] and not ocr_data['цена']:
                continue

            # 4. Поиск в БД
            match, score = hybrid_search(df, index, price_index, ocr_data, embed_model)

            if match and score >= MATCH_THRESHOLD:
                row = match["row"]
                sku = row.get("id_sku", "")
                product_name = row.get("product_name", "")

                # 5. Дедупликация
                is_duplicate = False
                if sku in recognized_products:
                    last_seen = recognized_products[sku]
                    if (frame_count - last_seen) < DUP_WINDOW:
                        is_duplicate = True
                    else:
                        recognized_products[sku] = frame_count
                else:
                    recognized_products[sku] = frame_count

                if not is_duplicate:
                    print(f"✅ НАЙДЕНО: {product_name[:50]}... | Score: {score:.2f}")

                    # Расчет времени в мс
                    timestamp_ms = int((frame_count / fps) * 1000)
                    x1, y1, x2, y2 = original_bbox

                    # Формирование строки результата как в sample.csv
                    result_row = {
                        "filename": os.path.basename(VIDEO_PATH),
                        "product_name": product_name,
                        "price_default": row.get("price_default", ""),
                        "price_card": row.get("price_card", ""),
                        "price_discount": row.get("price_discount", "нет"),
                        "barcode": row.get("barcode", ""),
                        "discount_amount": row.get("discount_amount", ""),
                        "id_sku": sku,
                        "print_datetime": row.get("print_datetime", ""),
                        "code": row.get("code", ""),
                        "additional_info": row.get("additional_info", ""),
                        "color": row.get("color", ""),
                        "special_symbols": row.get("special_symbols", ""),
                        "frame_timestamp": timestamp_ms,
                        "x_min": x1,
                        "y_min": y1,
                        "x_max": x2,
                        "y_max": y2,
                        "qr_code_barcode": row.get("qr_code_barcode", ""),
                        "price1_qr": row.get("price1_qr", ""),
                        "price2_qr": row.get("price2_qr", ""),
                        "price3_qr": row.get("price3_qr", ""),
                        "price4_qr": row.get("price4_qr", ""),
                        "wholesale_level_1_count": row.get("wholesale_level_1_count", ""),
                        "wholesale_level_1_price": row.get("wholesale_level_1_price", ""),
                        "wholesale_level_2_count": row.get("wholesale_level_2_count", ""),
                        "wholesale_level_2_price": row.get("wholesale_level_2_price", ""),
                        "action_price_qr": row.get("action_price_qr", ""),
                        "action_code_qr": row.get("action_code_qr", "")
                    }
                    results.append(result_row)
                else:
                    print(f"⏭️ Дубликат SKU {sku} (пропущено)")

        frame_count += 1

    cap.release()

    # Сохранение результатов
    if results:
        res_df = pd.DataFrame(results)
        # Переупорядочиваем колонки, чтобы frame_timestamp и координаты были в конце, как просили
        cols_order = [
            "filename", "product_name", "price_default", "price_card", "price_discount", "barcode",
            "discount_amount", "id_sku", "print_datetime", "code", "additional_info", "color",
            "special_symbols", "frame_timestamp", "x_min", "y_min", "x_max", "y_max",
            "qr_code_barcode", "price1_qr", "price2_qr", "price3_qr", "price4_qr",
            "wholesale_level_1_count", "wholesale_level_1_price", "wholesale_level_2_count",
            "wholesale_level_2_price", "action_price_qr", "action_code_qr"
        ]
        res_df = res_df[cols_order]
        res_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        print(f"\n📊 Готово! Сохранено {len(results)} уникальных товаров в {OUTPUT_CSV}")
    else:
        print("\n⚠️ Товары не найдены.")

if __name__ == "__main__":
    process_video()